In [ ]:
# ==============================================================================
# INFERENCIA, GRAD-CAM (5 PANELES) Y MATRICES DE CONFUSIÓN POR CLASE (GASTRO-AI)
# ==============================================================================
import os
import cv2
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b1
from PIL import Image
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_auc_score, precision_recall_curve
from google.colab import drive

# 1. Montar Google Drive y Configurar Dispositivo
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/GastroAI'
# Usar los pesos optimizados con Pérdida Enmascarada (Masked Loss)
WEIGHTS_PATH = os.path.join(BASE_DIR, 'best_gastroai_masked_loss.pth')
SPLITS_CSV = os.path.join(BASE_DIR, 'splits', 'patient_splits_5fold.csv')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Dispositivo activo: {device}")


In [ ]:
# ==============================================================================
# 2. ARQUITECTURA Y CARGA DE PESOS OPTIMIZADOS (MASKED LOSS)
# ==============================================================================
class GastroAI_EfficientNetB1(nn.Module):
    def __init__(self, num_classes=4):
        super(GastroAI_EfficientNetB1, self).__init__()
        self.backbone = efficientnet_b1(weights=None)
        in_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.3, inplace=True),
            nn.Linear(in_features, num_classes)
        )
        
    def forward(self, x):
        return self.backbone(x)

model = GastroAI_EfficientNetB1(num_classes=4).to(device)

if os.path.exists(WEIGHTS_PATH):
    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
    model.eval()
    print(f"✓ Pesos optimizados (Masked Loss) cargados exitosamente desde:\n  {WEIGHTS_PATH}")
else:
    alt_weights = os.path.join(BASE_DIR, 'best_gastroai_efficientnet_b1.pth')
    if os.path.exists(alt_weights):
        model.load_state_dict(torch.load(alt_weights, map_location=device))
        model.eval()
        print(f"⚠️ Usando pesos guardados alternativos desde: {alt_weights}")
    else:
        raise FileNotFoundError("❌ No se hallaron los pesos en Google Drive.")

target_cols = ['normal_mucosa', 'erythematous_gastritis', 'gastric_atrophy', 'h_pylori_signs']
class_labels_es = ['Mucosa Normal', 'Gastritis Eritematosa', 'Atrofia Gástrica', 'Signos H. Pylori']

val_transform = transforms.Compose([
    transforms.Resize((240, 240)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [ ]:
# ==============================================================================
# 3. GENERACIÓN DE MATRICES DE CONFUSIÓN (2x2) Y MÉTRICAS EVALUADAS (MASKED)
# ==============================================================================
df_splits = pd.read_csv(SPLITS_CSV)
val_df = df_splits[df_splits['fold'] == 0].reset_index(drop=True)

# Definir la máscara de evaluación real (Filtrar ceros imputados por falta de anotación en EGID)
def get_eval_masks(df):
    masks = []
    for _, row in df.iterrows():
        path_str = str(row['image_path']).lower()
        m = [1.0, 1.0, 1.0, 1.0]
        if 'egid' in path_str:
            if '1.h_pylori' in path_str or 'h_pylori_infection_status' in path_str:
                m[2] = 0.0  # Atrofia gástrica NO fue anotada en esta carpeta de EGID
            elif '2.presence_of_atrophy' in path_str or 'atrophy' in path_str:
                m[1] = 0.0  # Gastritis no anotada
                m[3] = 0.0  # H. Pylori no anotado
        masks.append(m)
    return np.array(masks, dtype=np.float32)

masks_val = get_eval_masks(val_df)
y_true_raw = val_df[target_cols].values
all_probs = []

print(f"🚀 Procesando inferencia sobre {len(val_df)} imágenes del conjunto de validación...")
with torch.no_grad():
    for idx, row in val_df.iterrows():
        try:
            img_pil = Image.open(row['image_path']).convert('RGB')
            input_tensor = val_transform(img_pil).unsqueeze(0).to(device)
            prob = torch.sigmoid(model(input_tensor)).cpu().numpy()
            all_probs.append(prob)
        except Exception:
            all_probs.append(np.array([[0.0, 0.0, 0.0, 0.0]]))

y_prob = np.vstack(all_probs)

# Calcular umbrales óptimos por clase únicamente sobre muestras válidas (Mask == 1)
optimal_thresholds = []
for i in range(4):
    valid_idx = (masks_val[:, i] == 1.0)
    if valid_idx.sum() > 0:
        precision, recall, thresholds = precision_recall_curve(y_true_raw[valid_idx, i], y_prob[valid_idx, i])
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
        best_idx = np.argmax(f1_scores)
        best_t = thresholds[best_idx] if best_idx < len(thresholds) else 0.50
        optimal_thresholds.append(best_t)
    else:
        optimal_thresholds.append(0.50)

optimal_thresholds = np.array(optimal_thresholds)

# Graficar Matrices de Confusión 2x2 exclusivamente sobre muestras anotadas
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.ravel()
metrics_summary = []

for i, name in enumerate(class_labels_es):
    valid_mask = (masks_val[:, i] == 1.0)
    yt = y_true_raw[valid_mask, i]
    yp_prob = y_prob[valid_mask, i]
    thresh = optimal_thresholds[i]
    yp_bin = (yp_prob >= thresh).astype(int)
    
    cm = confusion_matrix(yt, yp_bin)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Ausente (0)', 'Presente (1)'])
    disp.plot(ax=axes[i], cmap='Blues', values_format='d', colorbar=False)
    axes[i].set_title(f"Matriz de Confusión: {name}
(Evaluadas: {valid_mask.sum()} | Umbral = {thresh:.3f})", fontweight='bold', fontsize=11)
    
    auc_val = roc_auc_score(yt, yp_prob) if len(np.unique(yt)) > 1 else 0.5
    rep = classification_report(yt, yp_bin, output_dict=True, zero_division=0)
    
    metrics_summary.append({
        'Clase Patológica': name,
        'Umbral Óptimo': round(thresh, 3),
        'ROC-AUC (Masked)': round(auc_val, 4),
        'Sensibilidad (Recall)': round(rep['1']['recall'], 4),
        'Precisión': round(rep['1']['precision'], 4),
        'F1-Score': round(rep['1']['f1-score'], 4),
        'Muestras Reales Evaluadas': int(valid_mask.sum()),
        'Muestras Ignoradas (No Anotadas)': int((~valid_mask).sum())
    })

plt.suptitle("Matrices de Confusión de Gastro-AI (Filtrando Ceros Imputados de EGID)", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("
" + "="*80)
print(" 📋 REPORTE DE EVALUACIÓN CLÍNICA REAL Y MATRICES DE CONFUSIÓN")
print("="*80)
display(pd.DataFrame(metrics_summary))


In [ ]:
# ==============================================================================
# 4. EXPLICABILIDAD GRAD-CAM (5 PANELES) DIVERSIFICANDO MUESTRAS Y ETIQUETAS REALES
# ==============================================================================
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.target_layer.register_forward_hook(self._save_activation)
        self.target_layer.register_full_backward_hook(self._save_gradient)
        
    def _save_activation(self, module, input, output):
        self.activations = output
        
    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output
        
    def generate_heatmap(self, input_tensor, class_idx):
        self.model.eval()
        self.model.zero_grad()
        output = self.model(input_tensor)
        score = output[0, class_idx]
        score.backward(retain_graph=True)
        
        gradients = self.gradients.data.cpu().numpy()
        activations = self.activations.data.cpu().numpy()
        weights = np.mean(gradients, axis=(1, 2))
        cam = np.zeros(activations.shape[1:], dtype=np.float32)
        for i, w in enumerate(weights):
            cam += w * activations[i, :, :]
        cam = np.maximum(cam, 0)
        if np.max(cam) > 0:
            cam = cam / np.max(cam)
        probabilities = torch.sigmoid(output).detach().cpu().numpy()
        return cam, probabilities

gradcam = GradCAM(model, model.backbone.features[-1])

def explain_endoscopic_sample(img_path, df_splits, optimal_thresh):
    raw_pil = Image.open(img_path).convert('RGB')
    raw_resized = raw_pil.resize((240, 240))
    img_np = np.array(raw_resized)
    img_name = os.path.basename(img_path)
    
    row = df_splits[df_splits['image_path'] == img_path]
    targets_real = [
        int(row['normal_mucosa'].values[0]),
        int(row['erythematous_gastritis'].values[0]),
        int(row['gastric_atrophy'].values[0]),
        int(row['h_pylori_signs'].values[0])
    ] if len(row) > 0 else [None]*4
    
    # Verificar máscara para distinguir etiquetas reales vs ceros imputados
    path_str = str(img_path).lower()
    mask_eval = [1.0, 1.0, 1.0, 1.0]
    if 'egid' in path_str:
        if '1.h_pylori' in path_str or 'h_pylori_infection_status' in path_str:
            mask_eval[2] = 0.0  # Atrofia no anotada en esta carpeta de EGID
        elif '2.presence_of_atrophy' in path_str or 'atrophy' in path_str:
            mask_eval[1] = 0.0  # Gastritis no anotada
            mask_eval[3] = 0.0  # H. pylori no anotado

    input_tensor = val_transform(raw_pil).unsqueeze(0).to(device)
    fig, axes = plt.subplots(1, 5, figsize=(23, 5))
    
    axes[0].imshow(img_np)
    axes[0].set_title(f"1. Original
{img_name}", fontsize=10, fontweight='bold', color='navy')
    axes[0].axis('off')
    
    for c_idx in range(4):
        cam, probs = gradcam.generate_heatmap(input_tensor, c_idx)
        prob_pct = probs[0, c_idx] * 100
        opt_t = optimal_thresh[c_idx]
        pred_bin = 1 if (probs[0, c_idx] >= opt_t) else 0
        real_val = targets_real[c_idx]
        is_annotated = (mask_eval[c_idx] == 1.0)
        
        if is_annotated and real_val is not None:
            is_correct = (pred_bin == real_val)
            status_txt = f"Real: {real_val} | Pred: {prob_pct:.1f}%
" + ("✓ CORRECTO" if is_correct else "❌ ERROR")
            title_color = 'darkgreen' if is_correct else 'darkred'
        else:
            status_txt = f"Real: N/A (No Anotado) | Pred: {prob_pct:.1f}%
[Predicción Libre]"
            title_color = 'blue' if pred_bin == 1 else 'gray'
            
        heatmap = cv2.resize(cam, (240, 240))
        heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
        heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
        overlay = cv2.addWeighted(img_np, 0.6, heatmap_color, 0.4, 0)
        
        ax_i = c_idx + 1
        axes[ax_i].imshow(overlay)
        axes[ax_i].set_title(f"{ax_i+1}. {class_labels_es[c_idx]}
{status_txt}", 
                               fontsize=9, fontweight='bold', color=title_color)
        axes[ax_i].axis('off')
        
    plt.suptitle(f"Explicabilidad Grad-CAM - {img_name}", fontsize=13, fontweight='bold', y=1.03)
    plt.tight_layout()
    plt.show()

# Probar la inferencia sobre una imagen del conjunto de validación
sample_path = val_df.iloc[0]['image_path']
print(f"
🔍 Generando visualización de 5 paneles para: {sample_path}")
explain_endoscopic_sample(sample_path, val_df, optimal_thresholds)
